### 01 — HDX Philippines boundaries → Landing Volume

#### Goal

Preserve an HDX GeoJSON source version, prepare Spark-readable JSON Lines,
validate the source, record data-quality results, and publish a validated
handoff for notebook `03_hdx_landing_to_bronze`.

### Architecture

`HDX → Landing → Technical Staging → Validation → Bronze handoff`

#### Landing

Preserves:

- original downloaded ZIP
- extracted GeoJSON files
- source checksums
- source provenance
- immutable source version
- ingestion receipt

#### Technical staging

Converts the GeoJSON FeatureCollection into:

`one complete GeoJSON Feature per JSONL line`

This makes the source easier for Spark to process while preserving
the original source structure.

Technical staging is not a Medallion layer.

#### Validation

The notebook checks:

- required source layers
- checksums
- Spark parsing
- row-count reconciliation
- required source fields
- p-code completeness
- p-code uniqueness
- feature and geometry types
- administrative parent references

Validation results are recorded before the pipeline is allowed
to continue.

#### Configuration

Stable source settings come from:

`config/sources.yaml`

using:

`source_key = hdx_phl_admin`

Run-specific settings remain notebook widgets.

This notebook does not write Bronze records.
Notebook `03_hdx_landing_to_bronze` owns the Bronze load and Bronze rerun protection.

Run the cells in order and stop when a cell fails.

### Cell 2 — Configuration instructions

Cell 3 installs two pinned dependencies.

### ijson

Used to stream the large GeoJSON FeatureCollection without loading
the entire source file into Python memory.

### PyYAML

Used to read stable source configuration from:

`config/sources.yaml`

Cell 4 restarts Python after installation.

After the restart, continue with Cell 5.
Files, widgets, and Volume contents remain available.

In [0]:
%pip install ijson==3.4.0.post0 PyYAML==6.0.2

In [0]:
# Cell 4 — Restart Python after dependency installation

dbutils.library.restartPython()

## Cell 5 — Configure this ingestion run

Stable source configuration comes from:

`config/sources.yaml`

using the source key:

`hdx_phl_admin`

The YAML defines stable values such as:

- HDX dataset ID
- HDX service URL
- Landing root
- technical staging root
- required ADM levels
- GeoJSON filename rules
- future Bronze destination
- DQ audit table

Run-specific values remain notebook widgets because they may change
on every execution.

| Widget | Meaning |
| --- | --- |
| `source_key` | Source entry in `sources.yaml`; normally `hdx_phl_admin` |
| `config_path` | Optional explicit path to `sources.yaml`; leave blank for automatic discovery |
| `mode` | `download`, `uploaded`, or `existing` |
| `landing_version` | Required only when `mode=existing` |
| `uploaded_zip` | Required only when `mode=uploaded` |
| `resource_id` | Optional explicit HDX resource override |
| `execution_context` | `manual` or `job` |

### Normal new ingestion

Use:

`mode = download`

### Revalidate an already completed Landing version

Use:

`mode = existing`

and set `landing_version` to the complete immutable version directory.

Stable source configuration should not be changed just to rerun a batch.

In [0]:
# Cell 6 — Load source configuration and runtime parameters

from pathlib import Path, PurePosixPath
from urllib.parse import urlencode, urlparse
import re
import yaml


# =========================================================
# 1. Runtime parameters
# =========================================================
#
# Stable source configuration belongs in sources.yaml.
#
# Widgets below describe THIS execution only.
# =========================================================

RUNTIME_DEFAULTS = {
    "source_key": "hdx_phl_admin",
    "config_path": "",
    "mode": "download",
    "landing_version": "",
    "uploaded_zip": "",
    "resource_id": "",
    "execution_context": "manual",
}


for name, default in RUNTIME_DEFAULTS.items():
    dbutils.widgets.text(
        name,
        default,
        name.replace("_", " ").title(),
    )


runtime = {
    name: dbutils.widgets.get(name).strip()
    for name in RUNTIME_DEFAULTS
}


SOURCE_KEY = runtime["source_key"]

if not SOURCE_KEY:
    raise ValueError(
        "source_key cannot be blank."
    )


# =========================================================
# 2. Locate config/sources.yaml
# =========================================================

def resolve_config_path(explicit_value: str) -> Path:

    candidates = []

    # Explicit path has highest priority.
    if explicit_value:

        explicit_path = Path(explicit_value)

        if not explicit_path.is_absolute():
            explicit_path = Path.cwd() / explicit_path

        candidates.append(explicit_path)

    else:

        # Try current working directory and its parents.
        current = Path.cwd()

        for root in [current, *current.parents]:
            candidates.append(
                root / "config" / "sources.yaml"
            )

        # Databricks Workspace / Git folder fallback.
        try:
            notebook_path = (
                dbutils.notebook.entry_point
                .getDbutils()
                .notebook()
                .getContext()
                .notebookPath()
                .get()
            )

            workspace_notebook = (
                Path("/Workspace")
                / notebook_path.lstrip("/")
            )

            notebook_directory = (
                workspace_notebook.parent
            )

            for root in [
                notebook_directory,
                *notebook_directory.parents,
            ]:
                candidates.append(
                    root / "config" / "sources.yaml"
                )

        except Exception:
            pass


    checked = set()

    for candidate in candidates:

        candidate_string = str(candidate)

        if candidate_string in checked:
            continue

        checked.add(candidate_string)

        if candidate.is_file():
            return candidate


    raise FileNotFoundError(
        "Could not find config/sources.yaml. "
        "Set the config_path widget to the full path "
        "of the repository config/sources.yaml file."
    )


CONFIG_PATH = resolve_config_path(
    runtime["config_path"]
)


print(
    "Configuration file:",
    CONFIG_PATH,
)


# =========================================================
# 3. Read YAML
# =========================================================

with CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    configuration = yaml.safe_load(file)


if not isinstance(
    configuration,
    dict,
):
    raise ValueError(
        "sources.yaml must contain a YAML mapping."
    )


sources = configuration.get("sources")


if not isinstance(
    sources,
    dict,
):
    raise ValueError(
        "sources.yaml must contain a top-level "
        "'sources' mapping."
    )


if SOURCE_KEY not in sources:
    raise KeyError(
        f"Source key '{SOURCE_KEY}' was not found "
        f"in {CONFIG_PATH}."
    )


source_config = sources[SOURCE_KEY]


if not isinstance(
    source_config,
    dict,
):
    raise ValueError(
        f"Source '{SOURCE_KEY}' must contain "
        "a YAML mapping."
    )


# =========================================================
# 4. Validate HDX source contract
# =========================================================

REQUIRED_SOURCE_FIELDS = {
    "source_name",
    "source_system",
    "dataset_id",
    "source_format",
    "base_url",
    "landing_path",
    "staging_path",
    "required_levels",
    "layer_extensions",
    "layer_pattern",
    "bronze_table",
    "dq_results_table",
}


missing_fields = (
    REQUIRED_SOURCE_FIELDS
    - set(source_config)
)


if missing_fields:
    raise ValueError(
        f"Source '{SOURCE_KEY}' is missing "
        f"required fields: {sorted(missing_fields)}"
    )


if (
    str(
        source_config["source_format"]
    ).strip().lower()
    != "geojson"
):
    raise ValueError(
        "01_hdx_phl_to_landing expects "
        "source_format=GeoJSON."
    )


# =========================================================
# 5. Normalize source configuration
# =========================================================

raw_levels = source_config[
    "required_levels"
]


if isinstance(
    raw_levels,
    str,
):

    REQUIRED_LEVELS = sorted({
        int(value.strip())
        for value in raw_levels.split(",")
        if value.strip()
    })

elif isinstance(
    raw_levels,
    list,
):

    REQUIRED_LEVELS = sorted({
        int(value)
        for value in raw_levels
    })

else:

    raise ValueError(
        "required_levels must be a YAML list "
        "or comma-separated string."
    )


if (
    not REQUIRED_LEVELS
    or any(
        level < 0
        for level in REQUIRED_LEVELS
    )
):
    raise ValueError(
        "required_levels must contain "
        "nonnegative administrative levels."
    )


raw_extensions = source_config[
    "layer_extensions"
]


if isinstance(
    raw_extensions,
    str,
):

    LAYER_EXTENSIONS = sorted({
        value.strip().lower()
        for value in raw_extensions.split(",")
        if value.strip()
    })

elif isinstance(
    raw_extensions,
    list,
):

    LAYER_EXTENSIONS = sorted({
        str(value).strip().lower()
        for value in raw_extensions
        if str(value).strip()
    })

else:

    raise ValueError(
        "layer_extensions must be a YAML list "
        "or comma-separated string."
    )


if (
    not LAYER_EXTENSIONS
    or not set(
        LAYER_EXTENSIONS
    ).issubset({
        ".geojson",
        ".json",
    })
):
    raise ValueError(
        "layer_extensions may contain only "
        ".geojson and/or .json."
    )


# =========================================================
# 6. Build effective configuration
# =========================================================

settings = {

    # Source identity
    "source_key": SOURCE_KEY,
    "source_name": str(
        source_config["source_name"]
    ),
    "source_system": str(
        source_config["source_system"]
    ),

    # Stable YAML configuration
    "dataset_id": str(
        source_config["dataset_id"]
    ),
    "source_format": str(
        source_config["source_format"]
    ),
    "hdx_base_url": str(
        source_config["base_url"]
    ),
    "landing_path": str(
        source_config["landing_path"]
    ),
    "staging_path": str(
        source_config["staging_path"]
    ),
    "required_levels": REQUIRED_LEVELS,
    "layer_extensions": LAYER_EXTENSIONS,
    "layer_pattern": str(
        source_config["layer_pattern"]
    ),
    "bronze_table": str(
        source_config["bronze_table"]
    ),
    "dq_results_table": str(
        source_config["dq_results_table"]
    ),

    # Runtime parameters
    "mode": runtime["mode"],
    "landing_version": runtime[
        "landing_version"
    ],
    "uploaded_zip": runtime[
        "uploaded_zip"
    ],
    "resource_id": runtime[
        "resource_id"
    ],
    "execution_context": runtime[
        "execution_context"
    ],

    # Audit information
    "config_path": str(CONFIG_PATH),
}


SOURCE_NAME = settings[
    "source_name"
]

SOURCE_SYSTEM = settings[
    "source_system"
]

SOURCE_FORMAT = settings[
    "source_format"
]


# =========================================================
# 7. Operational constants
# =========================================================
#
# These describe HOW the program runs.
# They are not characteristics of the external source.
# =========================================================

MAX_ATTEMPTS = 3
REQUEST_TIMEOUT = 60
CHUNK_SIZE = 1024 * 1024
MAX_RETRY_DELAY = 60


# =========================================================
# 8. Validate Landing location
# =========================================================

landing_parts = PurePosixPath(
    settings["landing_path"]
).parts


if (
    len(landing_parts) < 5
    or landing_parts[:2]
    != ("/", "Volumes")
    or ".." in landing_parts
):
    raise ValueError(
        "landing_path must point inside an "
        "existing /Volumes/catalog/schema/volume."
    )


LANDING_ROOT = Path(
    settings["landing_path"]
)

VOLUME_ROOT = Path(
    *landing_parts[:5]
)


# =========================================================
# 9. Validate dataset configuration
# =========================================================

DATASET_ID = settings[
    "dataset_id"
]


if not re.fullmatch(
    r"[A-Za-z0-9_-]+",
    DATASET_ID,
):
    raise ValueError(
        "dataset_id must be a valid "
        "HDX dataset name or ID."
    )


base = urlparse(
    settings["hdx_base_url"]
)


if (
    base.scheme != "https"
    or not base.netloc
    or base.username
    or base.password
):
    raise ValueError(
        "base_url must be a public HTTPS URL."
    )


API_URL = (
    settings["hdx_base_url"].rstrip("/")
    + "/api/3/action/package_show?"
    + urlencode({
        "id": DATASET_ID
    })
)


# =========================================================
# 10. Runtime mode
# =========================================================

MODE = settings[
    "mode"
].lower()


if MODE not in {
    "download",
    "uploaded",
    "existing",
}:
    raise ValueError(
        "mode must be download, uploaded, "
        "or existing."
    )


REQUESTED_RESOURCE_ID = settings[
    "resource_id"
]

UPLOADED_ZIP = settings[
    "uploaded_zip"
]


# =========================================================
# 11. Layer matching rules
# =========================================================

LAYER_PATTERN = re.compile(
    settings["layer_pattern"],
    re.IGNORECASE,
)


if LAYER_PATTERN.groups != 1:
    raise ValueError(
        "layer_pattern must capture exactly "
        "one administrative-level number."
    )


# =========================================================
# 12. Target tables
# =========================================================

BRONZE_TABLE = settings[
    "bronze_table"
]

DQ_RESULTS_TABLE = settings[
    "dq_results_table"
]


def quoted_table(value):

    parts = value.split(".")

    if (
        len(parts) != 3
        or any(
            not re.fullmatch(
                r"[A-Za-z_][A-Za-z0-9_]*",
                part,
            )
            for part in parts
        )
    ):
        raise ValueError(
            "Use a catalog.schema.table name "
            "with simple identifiers."
        )

    return ".".join(
        f"`{part}`"
        for part in parts
    )


quoted_table(
    BRONZE_TABLE
)

quoted_table(
    DQ_RESULTS_TABLE
)


if (
    DQ_RESULTS_TABLE.lower()
    == BRONZE_TABLE.lower()
):
    raise ValueError(
        "The DQ table must differ from "
        "the geography Bronze table."
    )


# =========================================================
# 13. Extraction contract
# =========================================================

EXTRACTION_SPEC = {
    "source_format": SOURCE_FORMAT,
    "layer_extensions": LAYER_EXTENSIONS,
    "required_levels": REQUIRED_LEVELS,
    "layer_pattern": settings[
        "layer_pattern"
    ],
}


# =========================================================
# 14. Execution context
# =========================================================

EXECUTION_CONTEXT = (
    settings[
        "execution_context"
    ].lower()
)


if EXECUTION_CONTEXT not in {
    "manual",
    "job",
}:
    raise ValueError(
        "execution_context must be "
        "manual or job."
    )


VALIDATION_PASSED = False


# =========================================================
# 15. Configuration summary
# =========================================================

print(
    "Source key:",
    SOURCE_KEY,
)

print(
    "Source:",
    SOURCE_NAME,
)

print(
    "Source system:",
    SOURCE_SYSTEM,
)

print(
    "Dataset ID:",
    DATASET_ID,
)

print(
    "Source format:",
    SOURCE_FORMAT,
)

print(
    "Landing folder:",
    LANDING_ROOT,
)

print(
    "Required levels:",
    REQUIRED_LEVELS,
)

print(
    "Mode:",
    MODE,
)

print(
    "Bronze table:",
    BRONZE_TABLE,
)

print(
    "DQ audit table:",
    DQ_RESULTS_TABLE,
)

### Cell 7 — Check the Landing Volume

Cell 8 verifies that the configured Unity Catalog Volume is accessible.

The notebook may create subfolders inside the existing Volume,
but it does not create the catalog, schema, or Volume itself.

Fix Volume access or permission errors before continuing.

In [0]:
# Cell 8 — Check the existing Volume

if not VOLUME_ROOT.is_dir():
    raise RuntimeError(
        "Cannot access the configured Volume. "
        "Check the path, compute, and permissions."
    )


LANDING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Landing root ready:",
    LANDING_ROOT,
)

### Cell 9 — Define ingestion helpers

Cell 10 defines reusable helpers for:

- UTC timestamps
- SHA-256 checksums
- atomic JSON writes
- chunked HTTP downloads
- bounded retries
- rate-limit handling
- safe ZIP extraction
- administrative-layer selection
- existing-version verification
- operational stage logging

The ingestion expects exactly one GeoJSON layer for every configured
administrative level.

Completed Landing versions must pass checksum verification before reuse.

In [0]:
# Cell 10 — Helper functions

import hashlib
import http.client
import json
import random
import shutil
import time
import uuid
import zipfile

from datetime import (
    datetime,
    timezone,
)

from email.utils import (
    parsedate_to_datetime,
)

from urllib.error import (
    HTTPError,
    URLError,
)

from urllib.request import (
    Request,
    urlopen,
)


def utc_now():

    return (
        datetime.now(
            timezone.utc
        ).isoformat()
    )


def sha256_file(path):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as stream:

        for chunk in iter(
            lambda: stream.read(
                CHUNK_SIZE
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def save_json(
    path,
    data,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        path.name
        + ".partial"
    )

    temporary.write_text(
        json.dumps(
            data,
            indent=2,
        ),
        encoding="utf-8",
    )

    temporary.replace(
        path
    )


def download_file(
    url,
    destination,
):

    for attempt in range(
        MAX_ATTEMPTS
    ):

        try:

            request = Request(
                url,
                headers={
                    "User-Agent":
                    "capstone-hdx-ingestion/1.0"
                },
            )

            with (
                urlopen(
                    request,
                    timeout=REQUEST_TIMEOUT,
                ) as response,
                destination.open(
                    "wb"
                ) as output,
            ):

                expected = (
                    response.headers.get(
                        "Content-Length"
                    )
                )

                written = 0

                for chunk in iter(
                    lambda: response.read(
                        CHUNK_SIZE
                    ),
                    b"",
                ):

                    output.write(
                        chunk
                    )

                    written += len(
                        chunk
                    )


            if (
                expected is not None
                and written
                != int(expected)
            ):
                raise (
                    http.client.IncompleteRead(
                        b"",
                        max(
                            0,
                            int(expected)
                            - written,
                        ),
                    )
                )


            return


        except (
            URLError,
            TimeoutError,
            ConnectionError,
            http.client.HTTPException,
        ) as error:

            destination.unlink(
                missing_ok=True
            )


            if (
                isinstance(
                    error,
                    HTTPError,
                )
                and error.code
                not in {
                    429,
                    500,
                    502,
                    503,
                    504,
                }
            ):
                raise


            if (
                attempt
                == MAX_ATTEMPTS - 1
            ):
                raise


            pause = (
                2 ** (attempt + 1)
                + random.random()
            )


            if isinstance(
                error,
                HTTPError,
            ):

                retry_after = (
                    error.headers.get(
                        "Retry-After",
                        "",
                    )
                )


                if retry_after.isdigit():

                    pause = max(
                        pause,
                        int(
                            retry_after
                        ),
                    )


                elif retry_after:

                    try:

                        retry_date = (
                            parsedate_to_datetime(
                                retry_after
                            )
                        )

                        pause = max(
                            pause,
                            (
                                retry_date
                                - datetime.now(
                                    timezone.utc
                                )
                            ).total_seconds(),
                        )

                    except (
                        ValueError,
                        TypeError,
                        OverflowError,
                    ):
                        pass


            pause = min(
                pause,
                MAX_RETRY_DELAY,
            )


            print(
                "Temporary network failure. "
                f"Retry {attempt + 2}/"
                f"{MAX_ATTEMPTS} "
                f"in {pause:.1f}s."
            )


            time.sleep(
                pause
            )


def extract_selected_layers(
    archive_path,
    destination,
):

    files = []


    with zipfile.ZipFile(
        archive_path
    ) as archive:

        layers = [
            member
            for member
            in archive.infolist()

            if (
                not member.is_dir()

                and PurePosixPath(
                    member.filename
                ).suffix.lower()
                in LAYER_EXTENSIONS
            )
        ]


        for level in REQUIRED_LEVELS:

            matches = []


            for member in layers:

                match = (
                    LAYER_PATTERN.search(
                        PurePosixPath(
                            member.filename
                        ).stem
                    )
                )


                if (
                    match
                    and int(
                        match.group(1)
                    )
                    == level
                ):
                    matches.append(
                        member
                    )


            if len(matches) != 1:

                available = [
                    member.filename
                    for member in layers
                ]

                raise ValueError(
                    f"Expected exactly one ADM{level} "
                    "GeoJSON layer. "
                    "Review layer_pattern. "
                    f"Available: {available}"
                )


            member = matches[0]

            relative = PurePosixPath(
                member.filename
            )


            if (
                relative.is_absolute()
                or ".." in relative.parts
                or "\\"
                in member.filename
            ):
                raise ValueError(
                    "Unexpected archive path: "
                    f"{member.filename}"
                )


            target = (
                destination.joinpath(
                    *relative.parts
                )
            )


            target.parent.mkdir(
                parents=True,
                exist_ok=True,
            )


            digest = hashlib.sha256()


            with (
                archive.open(
                    member
                ) as input_file,
                target.open(
                    "wb"
                ) as output,
            ):

                for chunk in iter(
                    lambda:
                    input_file.read(
                        CHUNK_SIZE
                    ),
                    b"",
                ):

                    digest.update(
                        chunk
                    )

                    output.write(
                        chunk
                    )


            if (
                target.stat().st_size
                != member.file_size
            ):
                raise ValueError(
                    "Incomplete extracted file: "
                    f"{member.filename}"
                )


            files.append({
                "path": str(
                    relative
                ),
                "bytes":
                    member.file_size,
                "sha256":
                    digest.hexdigest(),
                "admin_level":
                    level,
            })


    return files


def validate_existing_version(
    version_dir,
    receipt,
    expected_hash,
):

    if (
        receipt.get("status")
        != "ready"

        or receipt.get(
            "file_hash"
        )
        != expected_hash
    ):
        raise ValueError(
            "Existing receipt is "
            "incomplete or mismatched."
        )


    if (
        receipt.get(
            "extraction_spec"
        )
        != EXTRACTION_SPEC
    ):
        raise ValueError(
            "Extraction settings differ "
            "from the completed version. "
            "Review before reuse."
        )


    levels = [
        item.get(
            "admin_level"
        )
        for item
        in receipt[
            "extracted_files"
        ]
    ]


    if (
        sorted(levels)
        != REQUIRED_LEVELS
    ):
        raise ValueError(
            "The receipt does not contain "
            "exactly the requested layers."
        )


    archive_name = receipt[
        "source_file"
    ]


    if (
        PurePosixPath(
            archive_name
        ).name
        != archive_name
    ):
        raise ValueError(
            "Unexpected archive filename "
            "in receipt."
        )


    if (
        sha256_file(
            version_dir
            / archive_name
        )
        != expected_hash
    ):
        raise ValueError(
            "Existing original ZIP failed "
            "its checksum check."
        )


    for item in receipt[
        "extracted_files"
    ]:

        relative = PurePosixPath(
            item["path"]
        )


        if (
            relative.is_absolute()
            or ".." in relative.parts
            or "\\"
            in item["path"]
        ):
            raise ValueError(
                "Unexpected extracted path "
                "in receipt."
            )


        target = (
            version_dir
            / "extracted"
            / item["path"]
        )


        if (
            not target.is_file()
            or sha256_file(
                target
            )
            != item["sha256"]
        ):
            raise ValueError(
                "Existing extracted file "
                "failed verification: "
                f"{item['path']}"
            )


def record_stage(
    stage,
    status,
    **details,
):

    save_json(
        RUN_DIR
        / f"{stage}.json",
        {
            "run_id": RUN_ID,
            "stage": stage,
            "status": status,
            "recorded_at":
                utc_now(),
            **details,
        },
    )

### Cell 11 — Select the source

Cell 12 starts a new ingestion run and records the effective configuration.

### Download mode

The notebook queries HDX metadata and automatically discovers the
configured GeoJSON ZIP.

### Uploaded mode

The notebook uses an existing ZIP stored under `/Volumes`.

### Existing mode

The notebook reuses an immutable Landing version only after verifying:

- receipt status
- extraction contract
- original ZIP checksum
- extracted GeoJSON checksums

The run configuration is written to the Landing `_runs` directory
for traceability.

In [0]:
# Cell 12 — Discover the source resource

from urllib.parse import unquote


RUN_ID = str(
    uuid.uuid4()
)

STARTED_AT = utc_now()

RUN_DIR = (
    LANDING_ROOT
    / "_runs"
    / RUN_ID
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


save_json(
    RUN_DIR
    / "configuration.json",
    {
        "source_key":
            SOURCE_KEY,

        "config_path":
            str(CONFIG_PATH),

        "effective_settings":
            settings,

        "operational": {
            "max_attempts":
                MAX_ATTEMPTS,

            "request_timeout_seconds":
                REQUEST_TIMEOUT,

            "chunk_size_bytes":
                CHUNK_SIZE,

            "max_retry_delay_seconds":
                MAX_RETRY_DELAY,
        },

        "recorded_at":
            STARTED_AT,
    },
)


try:

    # =====================================================
    # Download mode
    # =====================================================

    if MODE == "download":

        metadata_path = (
            RUN_DIR
            / "source_metadata.json"
        )


        download_file(
            API_URL,
            metadata_path,
        )


        payload = json.loads(
            metadata_path.read_text(
                encoding="utf-8"
            )
        )


        if not payload.get(
            "success"
        ):
            raise ValueError(
                "HDX API did not return success."
            )


        dataset_metadata = (
            payload["result"]
        )

        resources = (
            dataset_metadata[
                "resources"
            ]
        )


        for item in resources:

            print(
                item.get("name"),
                "|",
                item.get("format"),
                "|",
                item.get("id"),
            )


        candidates = [
            item
            for item in resources

            if (
                item.get(
                    "format",
                    "",
                ).strip().upper()
                == SOURCE_FORMAT.upper()

                and urlparse(
                    item.get(
                        "url",
                        "",
                    )
                ).path.lower().endswith(
                    ".zip"
                )
            )
        ]


        if REQUESTED_RESOURCE_ID:

            candidates = [
                item
                for item
                in candidates

                if item.get("id")
                == REQUESTED_RESOURCE_ID
            ]


        if len(candidates) != 1:
            raise ValueError(
                "Expected exactly one GeoJSON ZIP. "
                "Review the HDX resource list "
                "or resource_id parameter."
            )


        resource = candidates[0]

        download_url = (
            resource["url"]
        )


        if (
            urlparse(
                download_url
            ).scheme
            != "https"
        ):
            raise ValueError(
                "Expected an HTTPS "
                "source download."
            )


        source_file = unquote(
            PurePosixPath(
                urlparse(
                    download_url
                ).path
            ).name
        )


        print(
            "Selected resource:",
            resource.get("name"),
        )


    # =====================================================
    # Existing Landing mode
    # =====================================================

    elif MODE == "existing":

        value = settings[
            "landing_version"
        ]


        if not value:
            raise ValueError(
                "Existing mode requires "
                "landing_version."
            )


        version_dir = Path(
            value
        )


        if (
            version_dir.parent
            != LANDING_ROOT
            / DATASET_ID

            or not re.fullmatch(
                r"[a-f0-9]{64}",
                version_dir.name,
            )
        ):
            raise ValueError(
                "landing_version must be an "
                "immutable version directory under "
                "landing_path/dataset_id."
            )


        receipt_path = (
            version_dir
            / "receipt.json"
        )


        if not receipt_path.is_file():
            raise FileNotFoundError(
                f"Receipt not found: {receipt_path}"
            )


        receipt = json.loads(
            receipt_path.read_text(
                encoding="utf-8"
            )
        )


        if (
            receipt.get(
                "source_format"
            )
            != SOURCE_FORMAT

            or receipt.get(
                "dataset_id"
            )
            != DATASET_ID
        ):
            raise ValueError(
                "Select a ready GeoJSON receipt "
                "for the configured dataset."
            )


        print(
            "Verifying existing ZIP and "
            "GeoJSON checksums..."
        )


        validate_existing_version(
            version_dir,
            receipt,
            version_dir.name,
        )


        dataset_metadata = None


        resource = {
            "id":
                receipt.get(
                    "resource_id"
                ),

            "last_modified":
                receipt.get(
                    "source_last_modified"
                ),
        }


        download_url = (
            receipt.get(
                "source_url"
            )
        )

        source_file = (
            receipt[
                "source_file"
            ]
        )


        print(
            "Verified existing Landing version:",
            version_dir,
        )


    # =====================================================
    # Uploaded ZIP mode
    # =====================================================

    else:

        uploaded = Path(
            UPLOADED_ZIP
        )


        if (
            not UPLOADED_ZIP.startswith(
                "/Volumes/"
            )
            or not uploaded.is_file()
        ):
            raise ValueError(
                "Set uploaded_zip to an existing "
                "ZIP file under /Volumes/."
            )


        dataset_metadata = None

        resource = {
            "id":
                REQUESTED_RESOURCE_ID
                or None,
            "last_modified":
                None,
        }

        download_url = None

        source_file = (
            uploaded.name
        )


        print(
            "Using uploaded file:",
            uploaded,
        )


    # =====================================================
    # Common filename validation
    # =====================================================

    if (
        PurePosixPath(
            source_file
        ).name
        != source_file

        or "\\"
        in source_file

        or not source_file.lower().endswith(
            ".zip"
        )
    ):
        raise ValueError(
            "Expected a valid ZIP "
            "source filename."
        )


except Exception as error:

    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "failed",

            "stage":
                "source_discovery",

            "started_at":
                STARTED_AT,

            "finished_at":
                utc_now(),

            "error":
                str(error),
        },
    )

    raise

### Cell 13 — Preserve the immutable source version

Cell 14 downloads or stages the source ZIP and computes its SHA-256 hash.

The SHA-256 hash becomes the immutable source version.

Landing layout:

`landing_path / dataset_id / source_hash`

The version contains:

- original ZIP
- extracted GeoJSON layers
- receipt.json
- optional HDX source metadata

If the exact same ZIP bytes have already been landed successfully,
the notebook verifies and reuses the completed version instead of
creating another copy.

The original source is never modified.

In [0]:
# Cell 14 — Download, preserve, and extract

staged_zip = (
    RUN_DIR
    / "download.zip.partial"
)


try:

    # =====================================================
    # Existing version already verified
    # =====================================================

    if MODE == "existing":

        outcome = "reused"


    # =====================================================
    # Downloaded or uploaded source
    # =====================================================

    else:

        if MODE == "download":

            print(
                "Downloading GeoJSON ZIP..."
            )

            download_file(
                download_url,
                staged_zip,
            )


        else:

            shutil.copyfile(
                uploaded,
                staged_zip,
            )


        if not zipfile.is_zipfile(
            staged_zip
        ):
            raise ValueError(
                "Source content is not "
                "a valid ZIP archive."
            )


        file_hash = sha256_file(
            staged_zip
        )


        version_dir = (
            LANDING_ROOT
            / DATASET_ID
            / file_hash
        )


        receipt_path = (
            version_dir
            / "receipt.json"
        )


        # =================================================
        # Same source bytes already landed
        # =================================================

        if receipt_path.is_file():

            receipt = json.loads(
                receipt_path.read_text(
                    encoding="utf-8"
                )
            )


            validate_existing_version(
                version_dir,
                receipt,
                file_hash,
            )


            outcome = "reused"


            print(
                "Same verified source version. "
                "Reusing existing Landing files."
            )


        # =================================================
        # New source version
        # =================================================

        else:

            version_dir.mkdir(
                parents=True,
                exist_ok=True,
            )


            original_zip = (
                version_dir
                / source_file
            )


            if original_zip.exists():

                if (
                    sha256_file(
                        original_zip
                    )
                    != file_hash
                ):
                    raise ValueError(
                        "Existing original ZIP "
                        "has an unexpected checksum."
                    )


            else:

                temporary = (
                    original_zip.with_name(
                        original_zip.name
                        + ".partial"
                    )
                )


                shutil.copyfile(
                    staged_zip,
                    temporary,
                )


                temporary.replace(
                    original_zip
                )


            print(
                "Extracting required "
                "GeoJSON layers..."
            )


            extracted_files = (
                extract_selected_layers(
                    original_zip,
                    version_dir
                    / "extracted",
                )
            )


            if (
                dataset_metadata
                is not None
            ):
                save_json(
                    version_dir
                    / "source_metadata.json",
                    dataset_metadata,
                )


            receipt = {

                "status":
                    "ready",

                "receipt_schema_version":
                    3,

                # Source identity
                "source_key":
                    SOURCE_KEY,

                "source_name":
                    SOURCE_NAME,

                "source_system":
                    SOURCE_SYSTEM,

                "dataset_id":
                    DATASET_ID,

                "source_format":
                    SOURCE_FORMAT,

                # Source resource
                "resource_id":
                    resource.get(
                        "id"
                    ),

                "dataset_api_url":
                    API_URL,

                "source_url":
                    download_url,

                "uploaded_path":
                    (
                        UPLOADED_ZIP
                        if MODE
                        == "uploaded"
                        else None
                    ),

                "source_file":
                    source_file,

                "source_last_modified":
                    resource.get(
                        "last_modified"
                    ),

                # Ingestion identity
                "ingestion_timestamp":
                    STARTED_AT,

                "batch_id":
                    f"{DATASET_ID}:{file_hash}",

                "run_id":
                    RUN_ID,

                "file_hash":
                    file_hash,

                "source_version":
                    file_hash,

                "archive_bytes":
                    original_zip.stat().st_size,

                "ingestion_mode":
                    MODE,

                # Configuration used
                "config_path":
                    str(
                        CONFIG_PATH
                    ),

                "effective_configuration":
                    settings,

                # Extraction contract
                "extraction_spec":
                    EXTRACTION_SPEC,

                "extracted_files":
                    extracted_files,

                "completed_at":
                    utc_now(),
            }


            save_json(
                receipt_path,
                receipt,
            )


            outcome = "created"


    # =====================================================
    # Operational stage record
    # =====================================================

    record_stage(
        "landed",
        "ready",
        outcome=outcome,
        landing_receipt_path=str(
            receipt_path
        ),
        batch_id=receipt[
            "batch_id"
        ],
    )


    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "running",

            "outcome":
                outcome,

            "started_at":
                STARTED_AT,

            "updated_at":
                utc_now(),

            "version_path":
                str(
                    version_dir
                ),

            "batch_id":
                receipt[
                    "batch_id"
                ],
        },
    )


except Exception as error:

    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "failed",

            "stage":
                "download_or_extract",

            "started_at":
                STARTED_AT,

            "finished_at":
                utc_now(),

            "error":
                str(error),
        },
    )

    raise


finally:

    staged_zip.unlink(
        missing_ok=True
    )

### Cell 15 — Review the Landing output

Cell 16 prints the immutable Landing version that was selected or created.

A ready Landing receipt confirms that:

- the source ZIP was preserved
- the requested GeoJSON layers were extracted
- checksums were recorded
- source provenance was recorded

It does not yet mean every GeoJSON record is valid.

Record-level checks happen after the technical Spark preparation step.

In [0]:
# Cell 16 — Verify Landing output

print(
    "Landing version:",
    version_dir,
)

print(
    "Receipt:",
    receipt_path,
)

print(
    "Batch ID:",
    receipt[
        "batch_id"
    ],
)

print(
    "Source version:",
    receipt[
        "source_version"
    ],
)


for item in receipt[
    "extracted_files"
]:

    if (
        item["path"]
        .lower()
        .endswith(
            tuple(
                LAYER_EXTENSIONS
            )
        )
    ):

        print(
            "GeoJSON layer:",
            version_dir
            / "extracted"
            / item["path"],
        )


print(
    "Future Bronze destination:",
    BRONZE_TABLE,
)

### Cell 17 — Configure Spark technical staging

GeoJSON stores all records inside a FeatureCollection.

For Spark ingestion we prepare JSON Lines where:

`ONE LINE = ONE COMPLETE GeoJSON FEATURE`

The conversion:

- preserves properties
- preserves IDs
- preserves geometry
- preserves numeric tokens
- does not calculate centroids
- does not standardize business fields

The original GeoJSON remains unchanged in Landing.

A conversion specification is hashed so the same source version and
same conversion logic can safely reuse prepared output.

In [0]:
# Cell 18 — Prepare technical staging configuration

from pathlib import (
    Path,
    PurePosixPath,
)

from collections import (
    Counter,
)

from datetime import (
    datetime,
    timezone,
)

import hashlib
import json
import shutil
import uuid
import ijson


# =========================================================
# Conversion implementation version
# =========================================================

CONVERSION_VERSION = "1"

PARSER_BUFFER_BYTES = (
    64 * 1024
)

OUTPUT_TARGET_BYTES = (
    128
    * 1024
    * 1024
)

HASH_CHUNK_BYTES = (
    1024
    * 1024
)

EXPECTED_IJSON_VERSION = (
    "3.4.0.post0"
)


if (
    ijson.__version__
    != EXPECTED_IJSON_VERSION
):
    raise ValueError(
        "Run Cells 3–4 to install "
        "the expected ijson version."
    )


def utc_now():

    return (
        datetime.now(
            timezone.utc
        ).isoformat()
    )


def file_digest(path):

    digest = (
        hashlib.sha256()
    )


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda:
            file.read(
                HASH_CHUNK_BYTES
            ),
            b"",
        ):
            digest.update(
                chunk
            )


    return digest.hexdigest()


def write_json(
    path,
    data,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary = (
        path.with_name(
            path.name
            + ".partial"
        )
    )


    temporary.write_text(
        json.dumps(
            data,
            indent=2,
        ),
        encoding="utf-8",
    )


    temporary.replace(
        path
    )


def checked_relative(
    value,
):

    relative = PurePosixPath(
        value
    )


    if (
        relative.is_absolute()
        or ".." in relative.parts
        or "\\"
        in value
    ):
        raise ValueError(
            f"Unsafe relative path: {value}"
        )


    return relative


# =========================================================
# Re-open the authoritative Landing receipt
# =========================================================

SOURCE_VERSION_DIR = (
    version_dir
)


with (
    SOURCE_VERSION_DIR
    / "receipt.json"
).open(
    encoding="utf-8"
) as file:

    SOURCE_RECEIPT = (
        json.load(file)
    )


if (
    SOURCE_RECEIPT.get(
        "status"
    )
    != "ready"

    or SOURCE_RECEIPT.get(
        "source_format"
    )
    != "GeoJSON"
):
    raise ValueError(
        "Select a ready GeoJSON receipt."
    )


if (
    SOURCE_RECEIPT.get(
        "dataset_id"
    )
    != DATASET_ID
):
    raise ValueError(
        "Landing receipt dataset does not "
        "match the configured source."
    )


SOURCE_LAYERS = (
    SOURCE_RECEIPT[
        "extracted_files"
    ]
)


levels = [
    item.get(
        "admin_level"
    )
    for item
    in SOURCE_LAYERS
]


if (
    sorted(levels)
    != SOURCE_RECEIPT[
        "extraction_spec"
    ][
        "required_levels"
    ]
):
    raise ValueError(
        "Receipt layers do not match "
        "the extraction specification."
    )


for item in SOURCE_LAYERS:

    relative = (
        checked_relative(
            item["path"]
        )
    )


    if (
        relative.suffix.lower()
        not in {
            ".geojson",
            ".json",
        }
    ):
        raise ValueError(
            "Receipt contains an unexpected "
            "source file type."
        )


# =========================================================
# Staging root from YAML
# =========================================================

staging_value = settings[
    "staging_path"
]


staging_parts = (
    PurePosixPath(
        staging_value
    ).parts
)


if (
    len(staging_parts) < 5
    or staging_parts[:2]
    != ("/", "Volumes")
    or ".." in staging_parts
):
    raise ValueError(
        "staging_path must be inside "
        "an existing Volume."
    )


STAGING_ROOT = Path(
    staging_value
)


if not Path(
    *staging_parts[:5]
).is_dir():

    raise ValueError(
        "Cannot access the staging Volume."
    )


if (
    STAGING_ROOT
    == SOURCE_VERSION_DIR

    or SOURCE_VERSION_DIR
    in STAGING_ROOT.parents
):
    raise ValueError(
        "Use a staging folder outside "
        "the immutable source version directory."
    )


STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================
# Conversion specification
# =========================================================

CONVERSION_SPEC = {

    "converter_version":
        CONVERSION_VERSION,

    "ijson_version":
        ijson.__version__,

    "ijson_backend":
        ijson.backend,

    "parser_buffer_bytes":
        PARSER_BUFFER_BYTES,

    "output_target_bytes":
        OUTPUT_TARGET_BYTES,

    "numeric_handling":
        "decimal_tokens_without_float_conversion",

    "record_format":
        "one_complete_geojson_feature_per_line",
}


spec_hash = (
    hashlib.sha256(
        json.dumps(
            CONVERSION_SPEC,
            sort_keys=True,
        ).encode()
    ).hexdigest()
)


STAGING_VERSION = (
    STAGING_ROOT
    / SOURCE_RECEIPT[
        "dataset_id"
    ]
    / SOURCE_RECEIPT[
        "file_hash"
    ]
    / spec_hash
)


print(
    "Source version:",
    SOURCE_VERSION_DIR,
)

print(
    "Spark staging version:",
    STAGING_VERSION,
)

print(
    "Conversion spec hash:",
    spec_hash,
)

### Cell 19 — Prepare JSON Lines

Cell 20 streams GeoJSON tokens into JSON Lines.

Each output line contains one complete GeoJSON Feature.

The conversion does not load the complete FeatureCollection into Python memory.

Output files rotate near the configured target size, but a Feature is never
split across output files.

A ready staging manifest is published only after every required layer
has completed successfully.

On reruns:

- source checksums are verified
- prepared-file checksums are verified
- valid existing staging output is reused
- incomplete unpublished attempts are removed

In [0]:
# Cell 20 — Stream GeoJSON features into JSON Lines


class JsonEventWriter:

    """
    Write JSON tokens while retaining only
    JSON container state in memory.
    """

    def __init__(
        self,
        stream,
    ):

        self.stream = stream

        self.stack = []

        self.bytes_written = 0


    def emit(
        self,
        text,
    ):

        self.stream.write(
            text
        )

        self.bytes_written += len(
            text.encode(
                "utf-8"
            )
        )


    def event(
        self,
        event,
        value,
    ):

        # ---------------------------------------------
        # Object key
        # ---------------------------------------------

        if event == "map_key":

            parent = (
                self.stack[-1]
            )


            if not parent[
                "first"
            ]:
                self.emit(",")


            parent[
                "first"
            ] = False


            self.emit(
                json.dumps(
                    value,
                    ensure_ascii=False,
                )
                + ":"
            )

            return


        # ---------------------------------------------
        # Container end
        # ---------------------------------------------

        if event in {
            "end_map",
            "end_array",
        }:

            self.emit(
                "}"
                if event
                == "end_map"
                else "]"
            )

            self.stack.pop()

            return


        # ---------------------------------------------
        # Array separator
        # ---------------------------------------------

        if (
            self.stack
            and self.stack[-1][
                "kind"
            ]
            == "array"
        ):

            parent = (
                self.stack[-1]
            )


            if not parent[
                "first"
            ]:
                self.emit(",")


            parent[
                "first"
            ] = False


        # ---------------------------------------------
        # Container start
        # ---------------------------------------------

        if event in {
            "start_map",
            "start_array",
        }:

            self.emit(
                "{"
                if event
                == "start_map"
                else "["
            )


            self.stack.append({
                "kind":
                    "map"
                    if event
                    == "start_map"
                    else "array",

                "first":
                    True,
            })


        elif event == "number":

            self.emit(
                str(value)
            )


        elif event == "string":

            self.emit(
                json.dumps(
                    value,
                    ensure_ascii=False,
                )
            )


        elif event == "boolean":

            self.emit(
                "true"
                if value
                else "false"
            )


        elif event == "null":

            self.emit(
                "null"
            )


        else:

            raise ValueError(
                "Unsupported JSON event: "
                f"{event}"
            )


def convert_layer(
    source_path,
    level,
    output_directory,
    staging_directory,
):

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    outputs = []

    property_fields = set()

    geometry_types = Counter()


    count = 0

    part_number = 0

    part_bytes = 0


    collection_type = None

    source_crs = None

    features_array_seen = False


    output = None

    output_path = None

    writer = None


    feature_type = None

    geometry_type = None


    def finish_part():

        nonlocal output


        if (
            output is not None
            and not output.closed
        ):

            output.close()


            outputs.append({

                "path":
                    str(
                        output_path.relative_to(
                            staging_directory
                        )
                    ),

                "bytes":
                    output_path.stat().st_size,

                "sha256":
                    file_digest(
                        output_path
                    ),

                "admin_level":
                    level,
            })


    try:

        with source_path.open(
            "rb"
        ) as file:

            for (
                prefix,
                event,
                value,
            ) in ijson.parse(
                file,
                buf_size=PARSER_BUFFER_BYTES,
                use_float=False,
            ):

                # -------------------------------------
                # FeatureCollection metadata
                # -------------------------------------

                if (
                    prefix == "type"
                    and event == "string"
                ):
                    collection_type = value


                if (
                    prefix
                    == "crs.properties.name"
                    and event
                    == "string"
                ):
                    source_crs = value


                if (
                    prefix == "features"
                    and event
                    == "start_array"
                ):
                    features_array_seen = True


                # -------------------------------------
                # Start a Feature
                # -------------------------------------

                if (
                    prefix == "features.item"
                    and writer is None
                ):

                    if event != "start_map":
                        raise ValueError(
                            "Each features item must "
                            "be a GeoJSON Feature object."
                        )


                    if (
                        output is None
                        or part_bytes
                        >= OUTPUT_TARGET_BYTES
                    ):

                        finish_part()

                        part_number += 1


                        output_path = (
                            output_directory
                            / (
                                f"part-"
                                f"{part_number:05d}"
                                ".jsonl"
                            )
                        )


                        output = (
                            output_path.open(
                                "w",
                                encoding="utf-8",
                                newline="\n",
                            )
                        )


                        part_bytes = 0


                    writer = (
                        JsonEventWriter(
                            output
                        )
                    )


                    feature_type = None

                    geometry_type = (
                        "missing"
                    )


                # -------------------------------------
                # Write Feature tokens
                # -------------------------------------

                if writer is not None:

                    if (
                        prefix
                        == "features.item.type"
                        and event
                        == "string"
                    ):
                        feature_type = value


                    if (
                        prefix
                        == "features.item.geometry.type"
                        and event
                        == "string"
                    ):
                        geometry_type = value


                    if (
                        prefix
                        == "features.item.geometry"
                        and event
                        == "null"
                    ):
                        geometry_type = "null"


                    if (
                        prefix
                        == "features.item.properties"
                        and event
                        == "map_key"
                    ):
                        property_fields.add(
                            value
                        )


                    writer.event(
                        event,
                        value,
                    )


                    # ---------------------------------
                    # Feature completed
                    # ---------------------------------

                    if (
                        prefix == "features.item"
                        and event == "end_map"
                    ):

                        if (
                            feature_type
                            != "Feature"
                        ):
                            raise ValueError(
                                "A record does not have "
                                "type Feature."
                            )


                        writer.emit(
                            "\n"
                        )


                        part_bytes += (
                            writer.bytes_written
                        )


                        writer = None


                        geometry_types[
                            geometry_type
                        ] += 1


                        count += 1


                        if count % 100 == 0:

                            print(
                                f"ADM{level}: "
                                f"{count} features written..."
                            )


        if (
            collection_type
            != "FeatureCollection"

            or not features_array_seen

            or count == 0
        ):
            raise ValueError(
                "Expected a non-empty "
                "GeoJSON FeatureCollection."
            )


        finish_part()


        return (
            {
                "admin_level":
                    level,

                "rows":
                    count,

                "property_fields":
                    sorted(
                        property_fields
                    ),

                "geometry_types":
                    dict(
                        geometry_types
                    ),

                "source_crs":
                    source_crs
                    or (
                        "OGC:CRS84 "
                        "(GeoJSON default)"
                    ),
            },
            outputs,
        )


    finally:

        if (
            output is not None
            and not output.closed
        ):
            output.close()


def validate_prepared(
    manifest,
):

    if (
        manifest.get(
            "status"
        )
        != "ready"

        or manifest.get(
            "source_version"
        )
        != SOURCE_RECEIPT[
            "file_hash"
        ]

        or manifest.get(
            "conversion_spec"
        )
        != CONVERSION_SPEC
    ):
        raise ValueError(
            "Existing staging manifest is "
            "incomplete or mismatched."
        )


    if (
        sorted(
            layer[
                "admin_level"
            ]
            for layer
            in manifest[
                "layers"
            ]
        )
        != sorted(
            levels
        )
    ):
        raise ValueError(
            "Existing staging layers do not "
            "match the source receipt."
        )


    if (
        {
            item[
                "admin_level"
            ]
            for item
            in manifest[
                "files"
            ]
        }
        != set(
            levels
        )
    ):
        raise ValueError(
            "Existing staging files do not "
            "cover all source levels."
        )


    expected_inputs = {
        item[
            "admin_level"
        ]: (
            item[
                "path"
            ],
            item[
                "sha256"
            ],
        )
        for item
        in SOURCE_LAYERS
    }


    actual_inputs = {
        layer[
            "admin_level"
        ]: (
            layer[
                "source_file"
            ],
            layer[
                "source_sha256"
            ],
        )
        for layer
        in manifest[
            "layers"
        ]
    }


    if (
        actual_inputs
        != expected_inputs
    ):
        raise ValueError(
            "Existing staging inputs do not "
            "match the source receipt."
        )


    for item in manifest[
        "files"
    ]:

        relative = (
            checked_relative(
                item["path"]
            )
        )


        target = (
            STAGING_VERSION.joinpath(
                *relative.parts
            )
        )


        if (
            not target.is_file()

            or target.stat().st_size
            != item["bytes"]

            or file_digest(
                target
            )
            != item["sha256"]
        ):
            raise ValueError(
                "Existing staging file "
                "failed verification: "
                f"{item['path']}"
            )


# =========================================================
# Prepare / reuse staging version
# =========================================================

STAGING_VERSION.mkdir(
    parents=True,
    exist_ok=True,
)


PREPARED_MANIFEST_PATH = (
    STAGING_VERSION
    / "manifest.json"
)


conversion_run_id = str(
    uuid.uuid4()
)


conversion_log = (
    STAGING_VERSION
    / "_runs"
    / (
        f"{conversion_run_id}.json"
    )
)


attempt_dir = None


try:

    # =====================================================
    # Verify immutable Landing source files first
    # =====================================================

    for item in SOURCE_LAYERS:

        source_path = (
            SOURCE_VERSION_DIR
            / "extracted"
            / item["path"]
        )


        if (
            not source_path.is_file()

            or file_digest(
                source_path
            )
            != item[
                "sha256"
            ]
        ):
            raise ValueError(
                "Source checksum failed: "
                f"{item['path']}"
            )


    # =====================================================
    # Reuse existing verified conversion
    # =====================================================

    if (
        PREPARED_MANIFEST_PATH.is_file()
    ):

        PREPARED_MANIFEST = (
            json.loads(
                PREPARED_MANIFEST_PATH.read_text(
                    encoding="utf-8"
                )
            )
        )


        validate_prepared(
            PREPARED_MANIFEST
        )


        conversion_outcome = (
            "reused"
        )


        print(
            "Reusing verified JSON Lines files."
        )


    # =====================================================
    # Create conversion
    # =====================================================

    else:

        attempt_dir = (
            STAGING_VERSION
            / "attempts"
            / conversion_run_id
        )


        layer_results = []

        output_files = []


        for item in sorted(
            SOURCE_LAYERS,
            key=lambda value:
                value[
                    "admin_level"
                ],
        ):

            level = (
                item[
                    "admin_level"
                ]
            )


            print(
                f"Preparing ADM{level}: "
                f"{item['path']}"
            )


            result, files = (
                convert_layer(

                    SOURCE_VERSION_DIR
                    / "extracted"
                    / item[
                        "path"
                    ],

                    level,

                    attempt_dir
                    / f"adm{level}",

                    STAGING_VERSION,
                )
            )


            result.update(
                source_file=
                    item[
                        "path"
                    ],

                source_sha256=
                    item[
                        "sha256"
                    ],
            )


            layer_results.append(
                result
            )


            output_files.extend(
                files
            )


        PREPARED_MANIFEST = {

            "status":
                "ready",

            "dataset_id":
                SOURCE_RECEIPT[
                    "dataset_id"
                ],

            "source_version":
                SOURCE_RECEIPT[
                    "file_hash"
                ],

            "batch_id":
                SOURCE_RECEIPT[
                    "batch_id"
                ],

            "landing_version":
                str(
                    SOURCE_VERSION_DIR
                ),

            "conversion_spec":
                CONVERSION_SPEC,

            "layers":
                layer_results,

            "files":
                output_files,

            "completed_at":
                utc_now(),
        }


        write_json(
            PREPARED_MANIFEST_PATH,
            PREPARED_MANIFEST,
        )


        conversion_outcome = (
            "created"
        )


    write_json(
        conversion_log,
        {
            "status":
                "success",

            "run_id":
                conversion_run_id,

            "outcome":
                conversion_outcome,

            "finished_at":
                utc_now(),
        },
    )


except Exception as error:

    if (
        attempt_dir is not None
        and not PREPARED_MANIFEST_PATH.is_file()
    ):
        shutil.rmtree(
            attempt_dir,
            ignore_errors=True,
        )


    write_json(
        conversion_log,
        {
            "status":
                "failed",

            "run_id":
                conversion_run_id,

            "error":
                str(error),

            "finished_at":
                utc_now(),
        },
    )


    record_stage(
        "prepared",
        "failed",
        error=str(
            error
        ),
    )


    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "failed",

            "stage":
                "prepared",

            "error":
                str(error),

            "finished_at":
                utc_now(),
        },
    )


    raise


record_stage(
    "prepared",
    "ready",
    staging_manifest_path=str(
        PREPARED_MANIFEST_PATH
    ),
    outcome=conversion_outcome,
)


print(
    "Staging manifest:",
    PREPARED_MANIFEST_PATH,
)


for layer in PREPARED_MANIFEST[
    "layers"
]:

    print(
        f"ADM{layer['admin_level']}: "
        f"{layer['rows']} source features"
    )

### Cell 21 — Inspect records and source schema

Cell 22 parses the prepared JSON Lines using an explicit Spark schema.

The Bronze handoff preserves:

- complete GeoJSON Feature JSON
- source properties
- geometry
- feature ID

Geometry remains JSON text because Polygon and MultiPolygon coordinate
nesting differs.

Required source fields are checked without renaming or cleaning them.

Expected row counts come from the staging manifest, not hard-coded
ADM1/ADM2/ADM3 numbers.

This protects the pipeline when a future legitimate source version
contains a different number of administrative records.

In [0]:
# Cell 22 — Inspect GeoJSON records and collect checks

from pyspark.sql import (
    functions as F,
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    MapType,
)


FEATURE_SCHEMA = StructType([

    StructField(
        "type",
        StringType(),
    ),

    StructField(
        "id",
        StringType(),
    ),

    StructField(
        "properties",
        MapType(
            StringType(),
            StringType(),
        ),
    ),

    StructField(
        "geometry",
        StringType(),
    ),
])


inspection_frames = {}

inspection_checks = []

VALIDATION_PASSED = False

total_rows = 0


def make_check(
    level,
    name,
    kind,
    failed,
    total,
    severity="FAIL",
    detail="",
):

    return {
        "admin_level":
            level,

        "check_name":
            name,

        "check_type":
            kind,

        "status":
            (
                severity
                if failed
                else "PASS"
            ),

        "severity":
            severity,

        "fail_count":
            int(
                failed
            ),

        "total_count":
            int(
                total
            ),

        "fail_pct":
            (
                100.0
                * failed
                / total
                if total
                else 0.0
            ),

        "threshold":
            0.0,

        "detail":
            detail,
    }


source_levels = {
    layer[
        "admin_level"
    ]
    for layer
    in PREPARED_MANIFEST[
        "layers"
    ]
}


for layer in PREPARED_MANIFEST[
    "layers"
]:

    level = (
        layer[
            "admin_level"
        ]
    )

    expected_rows = (
        layer[
            "rows"
        ]
    )

    fields = set(
        layer[
            "property_fields"
        ]
    )


    required = {
        f"adm{level}_pcode",
        f"adm{level}_name",
    }


    if (
        level - 1
        in source_levels
    ):
        required.add(
            f"adm{level - 1}_pcode"
        )


    # =====================================================
    # Required source field contract
    # =====================================================

    for field in sorted(
        required
    ):

        inspection_checks.append(
            make_check(
                level,
                f"required_field:{field}",
                "SCHEMA",
                (
                    expected_rows
                    if field
                    not in fields
                    else 0
                ),
                expected_rows,
                detail=field,
            )
        )


    # =====================================================
    # Optional source center columns
    # =====================================================

    for field in (
        "center_lat",
        "center_lon",
    ):

        inspection_checks.append(
            make_check(
                level,
                f"optional_field:{field}",
                "SCHEMA",
                (
                    expected_rows
                    if field
                    not in fields
                    else 0
                ),
                expected_rows,
                severity="WARN",
                detail=field,
            )
        )


    paths = [
        str(
            STAGING_VERSION
            / item[
                "path"
            ]
        )
        for item
        in PREPARED_MANIFEST[
            "files"
        ]
        if item[
            "admin_level"
        ]
        == level
    ]


    try:

        frame = (
            spark.read
            .text(
                paths
            )

            .withColumn(
                "feature",
                F.from_json(
                    "value",
                    FEATURE_SCHEMA,
                    {
                        "mode":
                            "FAILFAST"
                    },
                ),
            )

            .select(
                F.col(
                    "value"
                ).alias(
                    "source_feature_json"
                ),
                "feature.*",
            )

            .withColumn(
                "geometry_type",
                F.get_json_object(
                    "geometry",
                    "$.type",
                ),
            )
        )


        geometry_counts = {
            (
                row[
                    "geometry_type"
                ]
                or "null"
            ):
                row["count"]

            for row
            in (
                frame
                .groupBy(
                    "geometry_type"
                )
                .count()
                .collect()
            )
        }


        rows = sum(
            geometry_counts.values()
        )


        inspection_checks.append(
            make_check(
                level,
                "spark_parse",
                "FORMAT",
                0,
                rows,
            )
        )


        inspection_checks.append(
            make_check(
                level,
                "manifest_row_count",
                "VOLUME",
                abs(
                    rows
                    - expected_rows
                ),
                max(
                    rows,
                    expected_rows,
                ),
                detail=(
                    f"expected={expected_rows}; "
                    f"actual={rows}"
                ),
            )
        )


        inspection_frames[
            level
        ] = frame


        total_rows += rows


        print(
            f"ADM{level}: "
            f"{rows} rows; "
            f"geometry types: "
            f"{geometry_counts}"
        )


        print(
            "Source fields:",
            sorted(
                fields
            ),
        )


        print(
            "Source CRS:",
            layer[
                "source_crs"
            ],
        )


    except Exception as error:

        inspection_checks.append(
            make_check(
                level,
                "spark_parse",
                "FORMAT",
                expected_rows,
                expected_rows,
                detail=str(
                    error
                )[:1000],
            )
        )


        print(
            f"ADM{level}: parsing failed. "
            "Run Cell 24 to record the failure."
        )


print(
    "Total inspected source features:",
    total_rows,
)

### Cell 23 — Validate records and persist DQ evidence

Cell 24 performs record-level source checks.

For each administrative layer it checks:

- nonblank p-code
- unique p-code
- nonblank source name
- GeoJSON `Feature` type
- Polygon / MultiPolygon geometry type
- parent administrative references

All checks are recorded before a FAIL stops the pipeline.

### PASS

The check met its expected rule.

### WARN

The condition is worth documenting but does not block ingestion.

### FAIL

The validated handoff is blocked.

DQ history is intentionally append-only even when the same immutable
source version is validated again.

The source rows themselves are never filtered or corrected here.
Business standardization belongs in Silver.

In [0]:
# Cell 24 — Validate records and persist audit results

from pyspark.sql.types import (
    IntegerType,
    LongType,
    DoubleType,
    TimestampType,
)


VALIDATION_PASSED = False


(
    RUN_DIR
    / "handoff.json"
).unlink(
    missing_ok=True
)


checks = list(
    inspection_checks
)


for (
    level,
    frame,
) in sorted(
    inspection_frames.items()
):

    expected = next(
        item["rows"]
        for item
        in PREPARED_MANIFEST[
            "layers"
        ]
        if item[
            "admin_level"
        ]
        == level
    )


    try:

        code_field = (
            f"adm{level}_pcode"
        )


        checked = (
            frame.withColumn(
                "checked_code",
                F.trim(
                    F.col(
                        "properties"
                    ).getItem(
                        code_field
                    )
                ),
            )
        )


        missing = (
            F.col(
                "checked_code"
            ).isNull()

            | (
                F.col(
                    "checked_code"
                )
                == ""
            )
        )


        source_name = (
            F.trim(
                F.col(
                    "properties"
                ).getItem(
                    f"adm{level}_name"
                )
            )
        )


        stats = checked.agg(

            F.count(
                "*"
            ).alias(
                "rows"
            ),


            F.sum(
                F.when(
                    missing,
                    1,
                ).otherwise(
                    0
                )
            ).alias(
                "missing_codes"
            ),


            F.sum(
                F.when(
                    source_name.isNull()
                    | (
                        source_name
                        == ""
                    ),
                    1,
                ).otherwise(
                    0
                )
            ).alias(
                "missing_names"
            ),


            F.sum(
                F.when(
                    F.col(
                        "type"
                    ).isNull()

                    | (
                        F.col(
                            "type"
                        )
                        != "Feature"
                    ),
                    1,
                ).otherwise(
                    0
                )
            ).alias(
                "bad_feature_types"
            ),


            F.sum(
                F.when(
                    F.col(
                        "geometry_type"
                    ).isNull()

                    | ~F.col(
                        "geometry_type"
                    ).isin(
                        "Polygon",
                        "MultiPolygon",
                    ),
                    1,
                ).otherwise(
                    0
                )
            ).alias(
                "bad_geometry_types"
            ),

        ).first()


        rows = (
            stats[
                "rows"
            ]
        )


        duplicates = (
            checked
            .filter(
                ~missing
            )
            .groupBy(
                "checked_code"
            )
            .count()
            .filter(
                F.col(
                    "count"
                )
                > 1
            )
            .agg(
                F.sum(
                    "count"
                ).alias(
                    "rows"
                )
            )
            .first()[
                "rows"
            ]
            or 0
        )


        for (
            check_name,
            check_type,
            failed,
        ) in [

            (
                "nonblank_pcode",
                "NULL",
                stats[
                    "missing_codes"
                ]
                or 0,
            ),

            (
                "unique_pcode",
                "UNIQUE",
                duplicates,
            ),

            (
                "nonblank_name",
                "NULL",
                stats[
                    "missing_names"
                ]
                or 0,
            ),

            (
                "feature_type",
                "ACCEPTED_VALUES",
                stats[
                    "bad_feature_types"
                ]
                or 0,
            ),

            (
                "polygon_type",
                "ACCEPTED_VALUES",
                stats[
                    "bad_geometry_types"
                ]
                or 0,
            ),
        ]:

            checks.append(
                make_check(
                    level,
                    check_name,
                    check_type,
                    failed,
                    rows,
                )
            )


        # =================================================
        # Parent-reference validation
        # =================================================

        parent_level = (
            level - 1
        )


        if (
            parent_level
            in source_levels
        ):

            if (
                parent_level
                not in inspection_frames
            ):

                checks.append(
                    make_check(
                        level,
                        "parent_reference",
                        "FOREIGN_KEY",
                        rows,
                        rows,
                        detail=(
                            "Parent layer could "
                            "not be inspected."
                        ),
                    )
                )


            else:

                parent_field = (
                    f"adm{parent_level}_pcode"
                )


                parents = (

                    inspection_frames[
                        parent_level
                    ]

                    .select(
                        F.trim(
                            F.col(
                                "properties"
                            ).getItem(
                                parent_field
                            )
                        ).alias(
                            "parent_code"
                        )
                    )

                    .filter(
                        F.col(
                            "parent_code"
                        ).isNotNull()

                        & (
                            F.col(
                                "parent_code"
                            )
                            != ""
                        )
                    )

                    .distinct()
                )


                children = (
                    frame.select(
                        F.trim(
                            F.col(
                                "properties"
                            ).getItem(
                                parent_field
                            )
                        ).alias(
                            "parent_code"
                        )
                    )
                )


                unmatched = (
                    children
                    .join(
                        parents,
                        "parent_code",
                        "left_anti",
                    )
                    .count()
                )


                checks.append(
                    make_check(
                        level,
                        "parent_reference",
                        "FOREIGN_KEY",
                        unmatched,
                        rows,
                    )
                )


    except Exception as error:

        checks.append(
            make_check(
                level,
                "record_checks_execution",
                "EXECUTION",
                expected,
                expected,
                detail=str(
                    error
                )[:1000],
            )
        )


# =========================================================
# Build audit records
# =========================================================

validation_id = str(
    uuid.uuid4()
)


executed_at = (
    utc_now()
)


audit_rows = [

    {
        "validation_id":
            validation_id,

        "run_id":
            RUN_ID,

        "batch_id":
            SOURCE_RECEIPT[
                "batch_id"
            ],

        "dataset":
            SOURCE_RECEIPT[
                "dataset_id"
            ],

        "executed_at":
            executed_at,

        "stage":
            "validated",

        **check,
    }

    for check
    in checks
]


report_path = (
    RUN_DIR
    / "dq_results.json"
)


save_json(
    report_path,
    {
        "validation_id":
            validation_id,

        "checks":
            audit_rows,
    },
)


# =========================================================
# Stable DQ audit schema
# =========================================================

DQ_SCHEMA = StructType([

    StructField(
        "validation_id",
        StringType(),
    ),

    StructField(
        "run_id",
        StringType(),
    ),

    StructField(
        "batch_id",
        StringType(),
    ),

    StructField(
        "dataset",
        StringType(),
    ),

    StructField(
        "executed_at",
        TimestampType(),
    ),

    StructField(
        "stage",
        StringType(),
    ),

    StructField(
        "admin_level",
        IntegerType(),
    ),

    StructField(
        "check_name",
        StringType(),
    ),

    StructField(
        "check_type",
        StringType(),
    ),

    StructField(
        "status",
        StringType(),
    ),

    StructField(
        "severity",
        StringType(),
    ),

    StructField(
        "fail_count",
        LongType(),
    ),

    StructField(
        "total_count",
        LongType(),
    ),

    StructField(
        "fail_pct",
        DoubleType(),
    ),

    StructField(
        "threshold",
        DoubleType(),
    ),

    StructField(
        "detail",
        StringType(),
    ),
])


# =========================================================
# Persist DQ history
# =========================================================

try:

    typed_audit_rows = [

        {
            **row,

            "executed_at":
                datetime.fromisoformat(
                    row[
                        "executed_at"
                    ]
                ),
        }

        for row
        in audit_rows
    ]


    audit_frame = (
        spark.createDataFrame(
            typed_audit_rows,
            DQ_SCHEMA,
        )
    )


    # Create only if it does not exist.
    (
        spark.createDataFrame(
            [],
            DQ_SCHEMA,
        )
        .write
        .format(
            "delta"
        )
        .mode(
            "ignore"
        )
        .saveAsTable(
            DQ_RESULTS_TABLE
        )
    )


    existing_schema = (
        spark.table(
            DQ_RESULTS_TABLE
        ).schema
    )


    expected_types = {
        field.name:
            field.dataType
        for field
        in DQ_SCHEMA
    }


    existing_types = {
        field.name:
            field.dataType
        for field
        in existing_schema
    }


    if (
        existing_types
        != expected_types
    ):
        raise ValueError(
            "Existing DQ table schema differs. "
            "Use a compatible dedicated table "
            "or migrate the schema explicitly."
        )


    (
        audit_frame
        .write
        .format(
            "delta"
        )
        .mode(
            "append"
        )
        .saveAsTable(
            DQ_RESULTS_TABLE
        )
    )


except Exception as error:

    record_stage(
        "validated",
        "failed",
        reason="audit_write",
        error=str(
            error
        ),
        report_path=str(
            report_path
        ),
    )


    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "failed",

            "stage":
                "audit_write",

            "error":
                str(error),

            "finished_at":
                utc_now(),
        },
    )


    raise


# =========================================================
# Gate the handoff
# =========================================================

problems = [
    (
        f"ADM{x['admin_level']} "
        f"{x['check_name']}"
    )
    for x in checks
    if x[
        "status"
    ]
    == "FAIL"
]


warnings = [
    item
    for item
    in checks
    if item[
        "status"
    ]
    == "WARN"
]


print(
    "DQ audit table:",
    DQ_RESULTS_TABLE,
)

print(
    "DQ JSON report:",
    report_path,
)


for check in checks:

    print(
        f"ADM{check['admin_level']} "
        f"{check['check_name']}: "
        f"{check['status']} "
        f"({check['fail_count']}/"
        f"{check['total_count']})"
    )


record_stage(
    "validated",
    (
        "failed"
        if problems
        else "ready"
    ),
    validation_id=
        validation_id,
    report_path=
        str(
            report_path
        ),
    fail_checks=
        len(
            problems
        ),
    warn_checks=
        len(
            warnings
        ),
)


if problems:

    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "failed",

            "stage":
                "validated",

            "fail_checks":
                problems,

            "finished_at":
                utc_now(),
        },
    )


    raise ValueError(
        "Checks were recorded. "
        "Review failures before Bronze: "
        + "; ".join(
            problems
        )
    )


VALIDATION_PASSED = True

VALIDATED_RUN_ID = RUN_ID


print(
    "Checks passed. "
    "Continue to Cell 26."
)

### Cell 25 — Publish the validated handoff

Cell 26 executes only after Cell 24 passes.

The handoff contains the exact validated artifacts that notebook 02
must consume.

This is important because notebook 02 should not guess which Landing
version or staging manifest is correct.

### Manual execution

The handoff is written to:

`_runs/<run_id>/handoff.json`

### Databricks Job execution

The same values are also published as Job task values.

A future Job can use:

`01_hdx_phl_to_landing`
↓
`02_hdx_landing_to_bronze`

with notebook 02 dependent on the successful completion of notebook 01.

Stages are owned separately:

- `landed` → source preserved
- `prepared` → JSONL staging ready
- `validated` → source checks passed
- `loaded` → belongs to notebook 02

Notebook 02 must independently enforce Bronze idempotency.

In [0]:
# Cell 26 — Publish validated paths for notebook 02


if (
    not globals().get(
        "VALIDATION_PASSED",
        False,
    )

    or globals().get(
        "VALIDATED_RUN_ID"
    )
    != RUN_ID
):
    raise ValueError(
        "Run Cell 24 successfully before "
        "publishing a handoff."
    )


# =========================================================
# Explicit contract between notebook 01 and notebook 02
# =========================================================

HANDOFF = {

    # Config/source identity
    "source_key":
        SOURCE_KEY,

    "dataset_id":
        SOURCE_RECEIPT[
            "dataset_id"
        ],

    "source_version":
        SOURCE_RECEIPT[
            "source_version"
        ],

    # Exact validated artifacts
    "landing_receipt_path":
        str(
            SOURCE_VERSION_DIR
            / "receipt.json"
        ),

    "staging_manifest_path":
        str(
            PREPARED_MANIFEST_PATH
        ),

    # Batch identity
    "batch_id":
        SOURCE_RECEIPT[
            "batch_id"
        ],

    # Downstream targets
    "bronze_table":
        BRONZE_TABLE,

    "dq_results_table":
        DQ_RESULTS_TABLE,

    # Validation evidence
    "validation_id":
        validation_id,
}


try:

    # =====================================================
    # Publish Job task values when orchestrated
    # =====================================================

    if (
        EXECUTION_CONTEXT
        == "job"
    ):

        for (
            key,
            value,
        ) in HANDOFF.items():

            dbutils.jobs.taskValues.set(
                key=key,
                value=value,
            )


    # =====================================================
    # Persist manual / operational handoff
    # =====================================================

    save_json(
        RUN_DIR
        / "handoff.json",
        HANDOFF,
    )


    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "success",

            "stage":
                "validated",

            "started_at":
                STARTED_AT,

            "finished_at":
                utc_now(),

            "batch_id":
                SOURCE_RECEIPT[
                    "batch_id"
                ],

            "handoff":
                HANDOFF,
        },
    )


except Exception as error:

    (
        RUN_DIR
        / "handoff.json"
    ).unlink(
        missing_ok=True
    )


    save_json(
        RUN_DIR
        / "run.json",
        {
            "run_id":
                RUN_ID,

            "status":
                "failed",

            "stage":
                "handoff",

            "error":
                str(
                    error
                ),

            "finished_at":
                utc_now(),
        },
    )


    raise


print(
    "Ready for 03_hdx_landing_to_bronze."
)

print(
    "Handoff file:",
    RUN_DIR
    / "handoff.json",
)

print(
    json.dumps(
        HANDOFF,
        indent=2,
    )
)